# Ski Pass Analysis

<div style="text-align: center;">
    
![Ski Lift, Art Neuveau](../public/ski_lift.jpg)
</div>

Ok, the point of this exercise is to take a look at the ski passes available and determine which one to buy for next year. Obviously the benefit to user/business is going to come down to price per day, the number of skiiable days per year, etc...


# Ski Pass Pricing

Ok, let's first take a look at the ski passes. This was obtained using the scraper in the parent library. Note that the providers use HEAVY bot moderation and these pages change frequently so the scrapers break. I need to just replace it with API calls to Gemini or OpenAI since they collect that info anyway.

```
Notes on Data: These prices do not reflect changes in prices due to pre-purchasing passes or using pro deals so uhhhhhhhhh treat with caution.
```

In [6]:
import pandas as pd

df = pd.read_csv('/media/kwierman/Data/powderpipeline/ski_passes.csv')

df

,Provider,Name,Age Range,Price
0,Epic,Epic Pass Adult,31,1089
1,Epic,Epic Pass Child,5-12,555
2,Epic,Epic Pass Teen,13-17,869
3,Epic,Epic Pass Young Adult,18-30,869
4,Ikon,Ikon Pass,23,1399
5,Ikon,Ikon Base Pass,23,949
6,Ikon,Ikon Session Pass,23,299
7,Indy,Learn to Turn,23,189
8,Indy,Indy Base Pass,23,368
9,Indy,Indy+ Pass,23,419


Ok, so the prices look pretty comparable. We'll use this later on.

# Ski Resort Overview

Let's take a look at a map of all the ski resorts and see where they are and which pass they're associated with

In [7]:
ski_resorts_df = pd.read_csv('/media/kwierman/Data/powderpipeline/ski_resorts.csv')
ski_resorts_df.head()


,Resort_Name,Pass_Affiliation,Latitude,Longitude,Base_Elevation_ft,Summit_Elevation_ft
0,Vail,Epic,39.6403,-106.3742,8120,11570
1,Breckenridge,Epic,39.4808,-106.0667,9600,12998
2,Keystone,Epic,39.6053,-105.9515,9280,12408
3,Crested Butte,Epic,38.8997,-106.9658,9375,12162
4,Beaver Creek,Epic,39.6042,-106.5165,8100,11440


In [8]:
import altair as alt
from vega_datasets import data
states = alt.topo_feature(data.us_10m.url, feature='states')

base_map = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    "albersUsa"
).properties(
    title='Ski Resorts in the United States',
    width=600,
    height=400
)

points = alt.Chart(ski_resorts_df).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color=alt.Color('Pass_Affiliation:N', scale=alt.Scale(scheme='viridis')), # Color by the specified column
    size=alt.value(50), # Set a fixed size for better visibility
    tooltip=['Resort_Name', 'Pass_Affiliation'] # Add tooltips for interactivity
)

# Combine the base map and the points
chart = base_map + points
chart.save('../public/ski_resorts.png')

# Display the final chart
chart

alt.LayerChart(...)

Ok, so obviously, the scraper didn't get _everything_, which is a bit dissappointing. I'll have to come back to that later. For now, let's focus on the ones that matter.

## Ski Resorts in Washington, Oregon, Alaska and Idaho

I _had_ planned on looking at Alaska. I'll have to redo that later. For now, let's take a look at what we've got.


In [9]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="powderpipeline_app")

def get_state_name(row):
    # Format the coordinates
    coordinates = f"{row['Latitude']}, {row['Longitude']}"
    # Use geolocator.reverse() to get location data
    location = geolocator.reverse(coordinates, exactly_one=True)
    # Extract the state name from the raw address dictionary
    if location and 'address' in location.raw:
        # 'state' is the key for state name in the Nominatim response
        return location.raw['address'].get('state', 'Unknown')
    else:
        return 'Unknown'
    # Add a delay to respect API rate limits (crucial for larger dataframes)
    time.sleep(10)

ski_resorts_df['State'] = ski_resorts_df.apply(get_state_name, axis=1)
ski_resorts_df.head()

GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=40.4572&lon=-106.8045&format=json&addressdetails=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)"))

In [ ]:
selected_states = ['Washington', 'Oregon', 'Idaho']
ski_resorts_df_local = ski_resorts_df[ ski_resorts_df['State'].isin(selected_states) ]

base_map = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    "albersUsa"
).properties(
    title='Ski Resorts Local to US',
    width=600,
    height=400
).transform_filter(
    alt.FieldOneOfPredicate(field='properties.name', oneOf=selected_states)
)

points = alt.Chart(ski_resorts_df_local).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color=alt.Color('Pass_Affiliation:N', scale=alt.Scale(scheme='viridis')), # Color by the specified column
    size=alt.value(50), # Set a fixed size for better visibility
    tooltip=['Resort_Name', 'Pass_Affiliation'] # Add tooltips for interactivity
)

# Combine the base map and the points
chart = base_map + points
chart.save('../public/ski_resorts.png')

# Display the final chart
chart

(76, 6)

# Snowfall over the past Decade

Ok, let's take a look at snowfall over the past decade. This is all to determine the amount of snow at each resort, and therefore the number of days each resort will be open and then the number of "good" days.


There is no single "magic number" of inches required for a ski resort to open. The minimum amount of snow needed varies drastically depending on the mountain's terrain, the density of the snow, and the resort's snowmaking capabilities.
Here is a general rule of thumb for how much compacted snow (the "base") is required[1]:
Grassy, smooth slopes (or heavy snowmaking): 12 to 18 inches[2][3]
Standard, mixed-terrain slopes: 18 to 24 inches[3][4]
Rocky, steep, or off-piste terrain: 36 to 48+ inches (3 to 4 feet)[1][4]
To give you an idea of how this looks across the country, here is a list of prominent U.S. ski resorts and the estimated minimum snow depth they need to start opening their lifts:

### 1. Arapahoe Basin (Colorado)
Minimum Snow Needed: ~18 to 20 inches[4]
Why: Known for being one of the first resorts to open in the U.S. (often in October), "A-Basin" uses a mix of high-altitude natural snow and snowmaking. Because the early-season High Noon trail is relatively smooth and heavily prepped, they can open their first runs with a packed base of just under two feet[4].
### 2. The Summit at Snoqualmie (Washington)
Minimum Snow Needed: 18 to 24 inches (Summit West) vs. 48+ inches (Alpental)[4][5]
Why: This resort is a perfect example of how terrain dictates snow requirements. The "Summit West" area consists of smooth, grassy slopes that have been brush-cut in the summer, meaning they only need about 1.5 to 2 feet of heavy Pacific Northwest snow to open[5]. However, their sister peak, Alpental, is incredibly steep and rocky, requiring at least 4 feet of snow before it is safe to ski[4][5].
### 3. Killington Resort (Vermont)
Minimum Snow Needed: 12 to 18 inches
Why: Dubbed the "Beast of the East," Killington relies heavily on massive, state-of-the-art snowmaking systems to open as early as November. Because man-made snow is much denser and more durable than natural powdery snow, they can pack down a highly durable 12-to-18-inch base on their initial grassy slopes to get the lifts spinning early[2][3].
### 4. Mountain High (California)
Minimum Snow Needed: ~12 inches[4]
Why: Located in Southern California, Mountain High relies heavily on man-made snow[3]. Their lower-mountain beginner and intermediate groomers are graded relatively smooth, allowing them to open early-season runs with as little as a foot of dense, heavily compacted machine-made snow[4].
### 5. Mammoth Mountain (California)
Minimum Snow Needed: 20 to 24 inches (Lower Mountain) vs. 40 to 60+ inches (Upper Mountain)[4]
Why: Mammoth can open its lower-elevation, heavily groomed trails with about two feet of snow[4]. However, the famous upper bowls are situated above the tree line and are notoriously rocky. To safely open the steep upper-mountain chutes without skiers destroying their equipment on jagged rocks, Mammoth often needs well over 4 to 5 feet of snow base.
### 6. Timberline Lodge / Mt. Hood (Oregon)
Minimum Snow Needed: 24 to 36 inches (Groomers) vs. 8+ feet (Off-Piste)[4]
Why: Located on a volcano, the terrain off the groomed trails is littered with massive boulders, crags, and tree stumps[4]. While the resort can open its heavily managed groomed runs with a few feet of snow, locals know that the off-piste freeride areas aren't truly safe to explore until the mountain receives 8 feet (96 inches) of base to entirely bury the volcanic rock[4].

## Key Factors That Determine Opening Day
If you are waiting for your local mountain to open, keep these three things in mind:
Snow Density: 20 inches of heavy, wet "Cascade Concrete" creates a much better base than 20 inches of light, dry "Champagne Powder." Grooming machines press the air out of dry snow, meaning 2 feet of fluffy powder might compress down to just 4 inches of actual base[5].
Man-Made vs. Natural: Resorts with strong snowmaking infrastructure need less natural snowfall, as man-made snow is basically tiny ice pellets that form an incredibly strong, durable foundation[2][3].
The "Sacrifice" Skis: When resorts open with the bare minimum (under 24 inches), the conditions are often called "early season conditions"[1][4]. This means there will likely be exposed rocks, dirt, and twigs, which is why veteran skiers use old, beat-up "rock skis" rather than brand-new equipment for the first few weeks of the season[6][7].